In [1]:
# Install requests if you don't have it
!pip install requests

import requests
import json
import re

print("✅ Setup complete")

✅ Setup complete


In [2]:
# Check if Ollama is accessible
try:
    response = requests.get("http://localhost:11434/api/tags")
    models = response.json()
    print("✅ Ollama is running!")
    print("Available models:", [m['name'] for m in models['models']])
except:
    print("❌ Ollama not running. Start it with: ollama serve")

✅ Ollama is running!
Available models: ['gpt-oss:20b', 'qwen2.5-coder:1.5b-base', 'nomic-embed-text:latest', 'llama3.1:8b']


In [5]:
import requests
import re

class OllamaAgent:
    def __init__(self, model="llama3.1:8b"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"
        self.memory = []
        
        # Verify model exists on init
        self._verify_model()
        
    def _verify_model(self):
        try:
            r = requests.get("http://localhost:11434/api/tags")
            models = [m['name'] for m in r.json().get('models', [])]
            if self.model not in models and f"{self.model}:latest" not in models:
                print(f"⚠️ Model '{self.model}' not found. Available: {models}")
                print(f"Run: ollama pull {self.model}")
        except Exception as e:
            print(f"⚠️ Cannot connect to Ollama: {e}")
        
    def call_llm(self, prompt):
        try:
            response = requests.post(self.url, json={
                "model": self.model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.7}
            }, timeout=30)
            
            data = response.json()
            
            if "error" in data:
                print(f"❌ Ollama error: {data['error']}")
                return "Thought: Error\nAction: FINAL ANSWER: Error"
                
            return data.get("response", "No response")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            return "Thought: Error\nAction: FINAL ANSWER: Error"
    
    def calculator(self, expression):
        try:
            result = eval(expression)
            return f"Result: {result}"
        except:
            return "Error: Invalid expression"
    
    def run(self, task, verbose=True):
        if verbose:
            print(f"🎯 Task: {task}\n")
        
        self.memory = []
        
        system_prompt = """You are a ReAct agent. Think step by step.
Available tools: calculator(expression)

Respond format:
Thought: [reasoning]
Action: [calculator(expression)] or [FINAL ANSWER: result]"""

        for step in range(10):
            context = "\n".join(self.memory)
            prompt = f"{system_prompt}\n\nTask: {task}\n{context}\nThought:"
            
            response = self.call_llm(prompt)
            
            if "Error" in response:
                return "Failed"
            
            # Parse
            thought = re.search(r"Thought:\s*(.+?)(?:\n|$)", response, re.IGNORECASE)
            action = re.search(r"Action:\s*(.+?)(?:\n|$)", response, re.IGNORECASE)
            
            if verbose:
                print(f"Step {step+1}: {response[:150]}...")
            
            if action:
                act = action.group(1).strip()
                
                if "FINAL ANSWER" in act.upper():
                    answer = act.split(":", 1)[-1].strip()
                    if verbose:
                        print(f"\n✅ Answer: {answer}")
                    return answer
                
                # Execute tool
                if "calculator" in act.lower():
                    expr = act.split("(")[1].split(")")[0].strip('"\'')
                    obs = self.calculator(expr)
                else:
                    obs = "Unknown tool"
                
                if verbose:
                    print(f"   Tool: {act} → {obs}")
                
                self.memory.extend([
                    f"Thought: {thought.group(1) if thought else '...'}",
                    f"Action: {act}",
                    f"Observation: {obs}"
                ])
        
        return "Max steps reached"

# Create and run
agent = OllamaAgent(model="llama3.1:8b")
result = agent.run("Calculate 15 * 23 + 47")

🎯 Task: Calculate 15 * 23 + 47

Step 1: First, I need to multiply 15 and 23.

Action: 15 * 23...
   Tool: 15 * 23 → Unknown tool
Step 2: Thought: The tool is a calculator that can handle expressions, which means I can use it to perform arithmetic operations and functions.

Action: [calc...
   Tool: [calculator(15*23)] → Result: 345
Step 3: Thought: Now that I have the result of the multiplication, I need to add 47 to it.

Action: [calculator(345 + 47)]...
   Tool: [calculator(345 + 47)] → Result: 392
Step 4: Thought: The task is complete, I have calculated the expression 15 * 23 + 47.
Action: FINAL ANSWER: 392...

✅ Answer: 392
